# Worked Example: Group-Level Statistics

## Goal
Run permutation OLS and a small synthetic multi-patient `time_resolved_mlm`
demo (sample data is single-patient).


In [ ]:
import numpy as np
import pandas as pd
from LFPAnalysis import statistics_utils

np.random.seed(42)
n = 80
df = pd.DataFrame({
    'power': np.random.randn(n) + 0.3 * np.linspace(-1, 1, n),
    'rpe': np.linspace(-1, 1, n) + 0.1 * np.random.randn(n),
})
ols_res = statistics_utils.permutation_regression_zscore(
    df, 'power ~ rpe', n_permutations=100
)
ols_res

In [ ]:
np.random.seed(0)
rng = np.random.default_rng(0)
participants = ['P01', 'P02', 'P03']
electrodes = {'P01': ['e1', 'e2'], 'P02': ['e1'], 'P03': ['e1', 'e2', 'e3']}
times = np.array([-0.2, 0.0, 0.2, 0.4])
rows = []
for p in participants:
    n_trials = 30
    rpe = rng.normal(size=n_trials)
    for elec in electrodes[p]:
        for trial in range(n_trials):
            for ts in times:
                signal = 0.4 * rpe[trial] * (ts > 0) + rng.normal(scale=1.0)
                rows.append({
                    'participant': p,
                    'unique_label': f'{p}_{elec}',
                    'trial': trial,
                    'ts': ts,
                    'tfr': signal,
                    'rpe': rpe[trial],
                })
smoothed_df = pd.DataFrame(rows)

mlm_res = statistics_utils.time_resolved_mlm(
    smoothed_df,
    y='tfr',
    formula='tfr ~ 1 + rpe',
    lower_group='unique_label',
    higher_group='participant',
    trial_key='trial',
    n_permutations=20,
)
print(mlm_res.head())

## Saving results

See chapter 15 (`15_saving_and_organizing_results`) for the recommended `results/` layout.

In [ ]:
# Uncomment to save. See chapter 15 for the recommended results/ layout.
# from pathlib import Path
# out = Path('../../results/worked-examples')
# out.mkdir(parents=True, exist_ok=True)
# ols_res.to_csv(out / 'perm_ols_rpe.csv', index=False)
# mlm_res.to_csv(out / 'mlm_time_resolved.csv', index=False)

## Next step

The save cell above demonstrates chapter 15's pattern. For more guidance, see chapter 15 (`15_saving_and_organizing_results`).